In [9]:
import pandas as pd
from docx import Document
from docx.shared import Pt
import os
import re
from docx.oxml.text.paragraph import CT_P
from docx.oxml.table import CT_Tbl

# --- 1. FILE CONFIGURATION ---
TEMPLATE_FILE = r"primar.docx"
STUDENT_EXCEL = r"STUDENT_EXCEL kitwe april 2026.xlsx"
MODULE_EXCEL = r"MODULE DETAILS (2).xlsx"

OUTPUT_FOLDER = r"Generated_Attendance_Sheets KITWE AUG 2026-END 1"
if not os.path.exists(OUTPUT_FOLDER):
    os.makedirs(OUTPUT_FOLDER)

# --- HELPER FUNCTION TO CLEAN DATA ---
def clean_value(value):
    val_str = str(value).strip()
    if not val_str:
        return ""
    if val_str.endswith('.0'):
        return val_str[:-2]
    return val_str

# --- REMOVE EMPTY PAGES ---
def remove_empty_pages(doc):
    for p in doc.paragraphs[::-1]:
        if p.text.strip() == "":
            p._element.getparent().remove(p._element)
        else:
            break

# --- 2. MAIN FUNCTION ---
def create_attendance_sheets():
    print("Loading Excel files...")
    try:
        df_students = pd.read_excel(STUDENT_EXCEL)
        df_modules = pd.read_excel(MODULE_EXCEL)

        df_students['SEMESTER'] = df_students['SEMESTER'].apply(clean_value)
        df_students['PROGRAM CODE'] = df_students['PROGRAM CODE'].apply(clean_value)

        df_modules['SEMESTER'] = df_modules['SEMESTER'].apply(clean_value)
        df_modules['PROGRAM CODE'] = df_modules['PROGRAM CODE'].apply(clean_value)

    except Exception as e:
        print(f"Error reading Excel files: {e}")
        return

    count = 0

    for index, row in df_modules.iterrows():
        try:
            mod_code = str(row['MODULE CODE'])
            mod_name = str(row['MODULE NAME'])
            prog_code = clean_value(row['PROGRAM CODE'])
            prog_name = str(row['PROGRAM NAME'])
            semester = clean_value(row['SEMESTER'])

            class_list = df_students[
                (df_students['PROGRAM CODE'] == prog_code) &
                (df_students['SEMESTER'] == semester)
            ]

            if class_list.empty:
                print(f"Skipping {mod_code}: No students found.")
                continue

            # Load Word Template
            doc = Document(TEMPLATE_FILE)

            # --- HEADER REPLACEMENT ---
            replacements = {
                r"Program\s*:\s*XX": f"Program : {prog_name}",
                r"SEM\s*:\s*XX": f"SEM : {semester}",
                r"C & N:\s*XX": f"C & N: {mod_code} - {mod_name}",
                r"Session &Time:\s*XX": "Session &Time:         HRs"
            }

            for paragraph in doc.paragraphs:
                text = paragraph.text
                for pattern, replacement in replacements.items():
                    text = re.sub(pattern, replacement, text, flags=re.IGNORECASE)
                paragraph.text = text

            # --- 4. TABLE PROCESSING ---
            table = doc.tables[0]
            total_data_rows = len(table.rows) - 1

            for idx, (_, student) in enumerate(class_list.iterrows()):
                if idx < total_data_rows:
                    row_cells = table.rows[idx + 1].cells
                else:
                    row_cells = table.add_row().cells

                row_cells[0].text = str(idx + 1)
                row_cells[1].text = str(student['STUDENT NO'])
                row_cells[2].text = str(student['STUDENT NAME'])
                row_cells[3].text = ""
                row_cells[4].text = ""

            # Clear blank rows
            for idx in range(len(class_list), total_data_rows):
                for cell in table.rows[idx + 1].cells:
                    cell.text = ""

            # --- 5. FIXED FONT SIZE AFTER TABLE ---
            body = doc._body._element
            elements = list(body)

            found_table = False

            for el in elements:
                if isinstance(el, CT_Tbl):
                    found_table = True
                    continue

                if found_table and isinstance(el, CT_P):
                    for p in doc.paragraphs:
                        if p._element == el:
                            for run in p.runs:
                                run.font.size = Pt(10)
                            break

            # --- 6. UPDATE TOTAL ---
            total_count = len(class_list)
            for paragraph in doc.paragraphs:
                if "TOTAL: …………….." in paragraph.text:
                    paragraph.text = paragraph.text.replace("TOTAL: ……………..", f"TOTAL: {total_count}           ")

            # --- 7. SAVE ---
            filename = f"{prog_name}_Sem{semester}_{mod_code}.docx".replace(" ", "")
            doc.save(os.path.join(OUTPUT_FOLDER, filename))

            print(f"Created: {filename}")
            count += 1

        except Exception as e:
            print(f"Error processing module {mod_code}: {e}")

    print(f"Finished! {count} Word files generated.")

if __name__ == "__main__":
    create_attendance_sheets()


Loading Excel files...
Created: B.EdPrimary_Sem1_770LA11.docx
Created: B.EdPrimary_Sem1_770IS12.docx
Created: B.EdPrimary_Sem1_770MA13.docx
Created: B.EdPrimary_Sem1_770HE14.docx
Created: B.EdPrimary_Sem1_770TS15.docx
Created: B.EdPrimary_Sem1_770SS16.docx
Skipping 770LA21: No students found.
Skipping 770IS22: No students found.
Skipping 770MA23: No students found.
Skipping 770HE24: No students found.
Skipping 770TS25: No students found.
Skipping 770SS26: No students found.
Created: B.EdPrimary_Sem3_770CS31.docx
Created: B.EdPrimary_Sem3_770IS32.docx
Created: B.EdPrimary_Sem3_770MA33.docx
Created: B.EdPrimary_Sem3_770HE34.docx
Created: B.EdPrimary_Sem3_770EA35.docx
Created: B.EdPrimary_Sem3_770SS36.docx
Created: B.EdPrimary_Sem3_770ED37.docx
Created: B.EdPrimary_Sem4_770ED41.docx
Created: B.EdPrimary_Sem4_770ED42.docx
Created: B.EdPrimary_Sem4_770ED43.docx
Created: B.EdPrimary_Sem4_770ED44.docx
Created: B.EdPrimary_Sem4_770EA45.docx
Created: B.EdPrimary_Sem4_770ED46.docx
Created: B.EdP

In [ ]:
import os
import win32com.client
from openpyxl import Workbook
# to find the documents which has more than 1 page
FOLDER_PATH = r"Generated_Attendance_Sheets KITWE AUG 2026-END"

word = win32com.client.Dispatch("Word.Application")
word.Visible = False

wb = Workbook()
ws = wb.active
ws.title = "Page Count"

ws["A1"] = "File Name"
ws["B1"] = "Page Count"

row = 2

print("Scanning folder:", FOLDER_PATH)

for root, dirs, files in os.walk(FOLDER_PATH):
    for file in files:
        if file.endswith(".docx") or file.endswith(".doc"):
            file_path = os.path.abspath(os.path.join(root, file))
            print("Processing:", file_path)

            try:
                doc = word.Documents.Open(file_path)
                pages = doc.ComputeStatistics(2)

                ws.cell(row=row, column=1, value=file)
                ws.cell(row=row, column=2, value=pages)

                doc.Close()
                row += 1

            except Exception as e:
                print(f"Error with {file}: {e}")

word.Quit()

wb.save("page_counts.xlsx")

print("Done!")

Scanning folder: Generated_Attendance_Sheets KITWE AUG 2026-END
Processing: C:\Users\USER\python\Generated_Attendance_Sheets KITWE AUG 2026-END\B.EdPrimary_Sem1_770HE14.docx
Processing: C:\Users\USER\python\Generated_Attendance_Sheets KITWE AUG 2026-END\B.EdPrimary_Sem1_770IS12.docx


In [ ]:
import os
import win32com.client

# to delete the 2nd page.
FOLDER_PATH = r"Generated_Attendance_Sheets mpika"

word = win32com.client.Dispatch("Word.Application")
word.Visible = False

for root, dirs, files in os.walk(FOLDER_PATH):
    for file in files:
        if file.endswith(".docx") or file.endswith(".doc"):

            file_path = os.path.abspath(os.path.join(root, file))
            print("Processing:", file_path)

            try:
                doc = word.Documents.Open(file_path)

                pages = doc.ComputeStatistics(2)

                if pages >= 2:
                    # Go to start of page 2
                    start_range = doc.GoTo(What=1, Which=1, Count=2)  # wdGoToPage
                    
                    # Go to start of page 3 (end of page 2)
                    try:
                        end_range = doc.GoTo(What=1, Which=1, Count=3)
                        doc.Range(start_range.Start, end_range.Start).Delete()
                    except:
                        # If page 3 doesn't exist → delete till end of doc
                        doc.Range(start_range.Start, doc.Content.End).Delete()

                    doc.Save()
                    print("Deleted page 2:", file)

                else:
                    print("Skipped (only 1 page):", file)

                doc.Close()

            except Exception as e:
                print(f"Error with {file}: {e}")

word.Quit()

print("Done!")

In [10]:
import os
import shutil
# step 2 arrage the files according to the program.
# Change this to your folder path
BASE_DIR = r"Generated_Attendance_Sheets KITWE AUG 2026-END 1"

for filename in os.listdir(BASE_DIR):
    if not filename.lower().endswith((".pdf", ".docx", ".doc", ".txt")):
        continue  # skip non-files if needed
    
    file_path = os.path.join(BASE_DIR, filename)
    
    if os.path.isfile(file_path):
        # Extract folder name → everything before "_Sem"
        if "_Sem" in filename:
            folder_name = filename.split("_Sem")[0]
        else:
            continue  # skip if pattern not found
        
        # Make folder if not exists
        folder_path = os.path.join(BASE_DIR, folder_name)
        os.makedirs(folder_path, exist_ok=True)
        
        # Move file
        shutil.move(file_path, os.path.join(folder_path, filename))
        print(f"Moved: {filename} → {folder_name}/")

print("Task Completed!")


Moved: B.COMNEWSYLLABUS_Sem4_551CO43.docx → B.COMNEWSYLLABUS/
Moved: B.COMNEWSYLLABUS_Sem4_551CO44.docx → B.COMNEWSYLLABUS/
Moved: B.COMNEWSYLLABUS_Sem4_551CO46.docx → B.COMNEWSYLLABUS/
Moved: B.COMNEWSYLLABUS_Sem4_551ES45.docx → B.COMNEWSYLLABUS/
Moved: B.COMNEWSYLLABUS_Sem4_551EX49.docx → B.COMNEWSYLLABUS/
Moved: B.COMNEWSYLLABUS_Sem4_551LA41.docx → B.COMNEWSYLLABUS/
Moved: B.COMNEWSYLLABUS_Sem4_551MG42.docx → B.COMNEWSYLLABUS/
Moved: B.EdPrimary_Sem1_770HE14.docx → B.EdPrimary/
Moved: B.EdPrimary_Sem1_770IS12.docx → B.EdPrimary/
Moved: B.EdPrimary_Sem1_770LA11.docx → B.EdPrimary/
Moved: B.EdPrimary_Sem1_770MA13.docx → B.EdPrimary/
Moved: B.EdPrimary_Sem1_770SS16.docx → B.EdPrimary/
Moved: B.EdPrimary_Sem1_770TS15.docx → B.EdPrimary/
Moved: B.EdPrimary_Sem3_770CS31.docx → B.EdPrimary/
Moved: B.EdPrimary_Sem3_770EA35.docx → B.EdPrimary/
Moved: B.EdPrimary_Sem3_770ED37.docx → B.EdPrimary/
Moved: B.EdPrimary_Sem3_770HE34.docx → B.EdPrimary/
Moved: B.EdPrimary_Sem3_770IS32.docx → B.EdPri

In [11]:
import os
import re
import shutil
# step 3 arrage the file according to the semester
ROOT_FOLDER = r"Generated_Attendance_Sheets KITWE AUG 2026-END 1"

# Regex to capture semester (Sem1, Sem2, etc.)
pattern = re.compile(r"(Sem\d+)")

for root, dirs, files in os.walk(ROOT_FOLDER):
    for file in files:
        file_path = os.path.join(root, file)

        # Skip if already inside a Sem folder
        if re.search(r"Sem\d+", root):
            continue

        # Find semester in filename
        match = pattern.search(file)

        if match:
            sem_folder = match.group(1)  # e.g., Sem2

            # Create Sem folder INSIDE the current folder
            dest_folder = os.path.join(root, sem_folder)
            os.makedirs(dest_folder, exist_ok=True)

            # Move file into that folder
            try:
                shutil.move(file_path, os.path.join(dest_folder, file))
                print(f"Moved: {file} → {root}\\{sem_folder}")
            except Exception as e:
                print(f"Error moving {file}: {e}")

print("Done organizing per folder!")

Moved: B.COMNEWSYLLABUS_Sem4_551CO43.docx → Generated_Attendance_Sheets KITWE AUG 2026-END 1\B.COMNEWSYLLABUS\Sem4
Moved: B.COMNEWSYLLABUS_Sem4_551CO44.docx → Generated_Attendance_Sheets KITWE AUG 2026-END 1\B.COMNEWSYLLABUS\Sem4
Moved: B.COMNEWSYLLABUS_Sem4_551CO46.docx → Generated_Attendance_Sheets KITWE AUG 2026-END 1\B.COMNEWSYLLABUS\Sem4
Moved: B.COMNEWSYLLABUS_Sem4_551ES45.docx → Generated_Attendance_Sheets KITWE AUG 2026-END 1\B.COMNEWSYLLABUS\Sem4
Moved: B.COMNEWSYLLABUS_Sem4_551EX49.docx → Generated_Attendance_Sheets KITWE AUG 2026-END 1\B.COMNEWSYLLABUS\Sem4
Moved: B.COMNEWSYLLABUS_Sem4_551LA41.docx → Generated_Attendance_Sheets KITWE AUG 2026-END 1\B.COMNEWSYLLABUS\Sem4
Moved: B.COMNEWSYLLABUS_Sem4_551MG42.docx → Generated_Attendance_Sheets KITWE AUG 2026-END 1\B.COMNEWSYLLABUS\Sem4
Moved: B.EdPrimary_Sem1_770HE14.docx → Generated_Attendance_Sheets KITWE AUG 2026-END 1\B.EdPrimary\Sem1
Moved: B.EdPrimary_Sem1_770IS12.docx → Generated_Attendance_Sheets KITWE AUG 2026-END 1\B.